IMPORTS

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, precision_score, recall_score

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
import seaborn as sns

LOAD DATA

In [2]:
data = pd.read_csv('../data/cleaned_books.csv')

# Basic cleaning
data['Description'] = data['Description'].fillna('')
data['Genre'] = data['Ranks and Genre'].apply(
    lambda x: x.split(',')[0] if isinstance(x, str) else ''
)

data.head()

,Book Name,Author,Number of Reviews_x,Price_x,Number of Reviews_y,Price_y,Description,Listening Time,Ranks and Genre,Rating,Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,313.0,10080.0,371.0,10080,"Over the past three years, Jay Shetty has beco...",10 hours and 54 minutes,",#1 in Audible Audiobooks & Originals (See Top...",4.9,
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,3658.0,615.0,3682.0,615,Brought to you by Penguin.,3 hours and 23 minutes,",#2 in Audible Audiobooks & Originals (See Top...",4.6,
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,20174.0,10378.0,20306.0,10378,"In this generation-defining self-help guide, a...",5 hours and 17 minutes,",#3 in Audible Audiobooks & Originals (See Top...",4.4,
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,4614.0,888.0,4678.0,888,Brought to you by Penguin.,5 hours and 35 minutes,",#5 in Audible Audiobooks & Originals (See Top...",4.6,
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,4302.0,1005.0,4308.0,1005,"Stop going through life, Start growing throug...",6 hours and 25 minutes,",#6 in Audible Audiobooks & Originals (See Top...",4.6,


FEATURE ENGINEERING (IMPORTANT)

In [3]:
data['combined_features'] = (
    data['Genre'] + " " +
    data['Author'] + " " +
    data['Description']
)

TF-IDF + COSINE SIMILARITY

In [4]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(data['combined_features'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

CONTENT-BASED MODEL

In [5]:
def get_content_recommendations(book_name, top_n=5):

    matches = data[data['Book Name'].str.contains(book_name, case=False, na=False)]

    if matches.empty:
        return []

    idx = matches.index[0]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]

    indices = [i[0] for i in sim_scores]

    return data.iloc[indices][['Book Name','Author','Rating']]




CLUSTERING MODEL

In [6]:
kmeans = KMeans(n_clusters=5, random_state=42)
data['Cluster'] = kmeans.fit_predict(tfidf_matrix)

def get_cluster_recommendations(book_name):

    matches = data[data['Book Name'].str.contains(book_name, case=False, na=False)]

    if matches.empty:
        return []

    idx = matches.index[0]
    cluster_id = data.loc[idx, 'Cluster']

    return data[data['Cluster'] == cluster_id].head(5)

SIMPLE COLLABORATIVE FILTERING (VALID VERSION)

In [7]:
# Simulated user-book matrix
user_item = pd.pivot_table(
    data,
    index='Author',
    columns='Book Name',
    values='Rating'
).fillna(0)

knn = NearestNeighbors(metric='cosine')
knn.fit(user_item)

def collaborative_recommendation(book_name):

    if book_name not in user_item.columns:
        return []

    book_vector = user_item[book_name].values.reshape(1, -1)

    distances, indices = knn.kneighbors(book_vector, n_neighbors=5)

    return user_item.columns[indices.flatten()]

HYBRID MODEL

In [8]:
def hybrid_recommendation(book_name):

    content = get_content_recommendations(book_name)
    cluster = get_cluster_recommendations(book_name)

    combined = pd.concat([content, cluster]).drop_duplicates()

    return combined.head(5)

HIDDEN GEMS

In [9]:
hidden_gems = data[
    (data['Rating'] > 4.5) &
    (data['Number of Reviews_x'] < 500)
]

hidden_gems[['Book Name','Author','Rating']].head()

,Book Name,Author,Rating
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,4.9
13,The Sandman,Neil Gaiman,5.0
17,Sherlock Holmes: The Definitive Collection,Arthur Conan Doyle,5.0
77,Wings of Fire,APJ Abdul Kalam,4.6
80,Sapiens,Yuval Noah Harari,4.6


TEST MODEL

In [10]:
book = "Atomic Habits"

print("Content-Based:")
print(get_content_recommendations(book))

print("\nCluster-Based:")
print(get_cluster_recommendations(book))

print("\nHybrid:")
print(hybrid_recommendation(book))

Content-Based:
                                              Book Name          Author  \
2497      Camp Jupiter Classified: A Probatio's Journal    Rick Riordan   
1617                     Tripwire: Jack Reacher, Book 3       Lee Child   
2173                   Die Trying: Jack Reacher, Book 2       Lee Child   
1016                                      The Overstory  Richard Powers   
633   The Great Influenza: The Story of the Deadlies...   John M. Barry   

      Rating  
2497     4.8  
1617     4.5  
2173     4.4  
1016     4.4  
633      4.6  

Cluster-Based:
                                           Book Name              Author  \
0  Think Like a Monk: The Secret of How to Harnes...          Jay Shetty   
1  Ikigai: The Japanese Secret to a Long and Happ...       Héctor García   
3  Atomic Habits: An Easy and Proven Way to Build...         James Clear   
8                   The Intelligent Investor Rev Ed.     Benjamin Graham   
9  Rich Dad Poor Dad: What the Rich Teach Their 

EVALUATION (REAL)

In [11]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

# Example RMSE (simple simulation)
y_true = test['Rating']
y_pred = np.full(len(test), train['Rating'].mean())

rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print("RMSE:", rmse)

RMSE: 1.554415521107682


PRECISION & RECALL (SIMPLIFIED)

In [12]:
# Define relevance threshold
threshold = 4.0

y_true_binary = (test['Rating'] >= threshold).astype(int)
y_pred_binary = (y_pred >= threshold).astype(int)

precision = precision_score(y_true_binary, y_pred_binary)
recall = recall_score(y_true_binary, y_pred_binary)

print("Precision:", precision)
print("Recall:", recall)

Precision: 0.0
Recall: 0.0


c:\Users\DELL\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
